# <font color='red'>
---
## <center> <font color='red'> Masters in Mathematical Finance
#### <center> <font color='red'> 2024 / 26
# <center> <font color='red'> Masters' Final Work
---
# <center> <font color='red'><font> Student: Petr Terletskiy </font>
### <center> <font color='red'><font> Number: l63023 </font>
---
##### <center>  <font color='red'><font> Project on Bitcoin's Volatility and Option Pricing Model Selection</font>

---

# Dependencies

In [1]:
%pip install yfinance coinmetrics.api_client --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import requests

import yfinance as yf
from coinmetrics.api_client import CoinMetricsClient

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
#plt.style.use('seaborn-v0_8')
#sns.set_palette("husl")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


# Data Retrieval

`btc_spot`: data, preço, volume

`btc_options`: data, strike, maturity, preço_compra, preço_venda, volume, etc.

`btc_on_chain`: data, transacoes_unicas, taxa_hash, dificuldade, volume_usd, idade_media_moedas, percentagem_inativas, fluxo_exchange, open_interest, funding_rate

## OHLCV Data

In [ ]:
# Download BTC ohlcv data from Yahoo Finance
ticker = 'BTC-USD'
btc_data_raw = yf.download(ticker, start="2014-09-17", end="2026-01-17", interval="1d")
btc_data_raw.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
print(f"\nData available: {btc_data_raw.shape[0]} days and {btc_data_raw.shape[1]} features.")
print(f"Period: {btc_data_raw.index[0].date()} to {btc_data_raw.index[-1].date()}")
btc_data_raw.head(3)

[*********************100%***********************]  1 of 1 completed


Data available: 4130 days and 5 features.
Period: 2014-09-17 to 2026-01-06


,Open,High,Low,Close,Volume
Date,,,,,
2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800
2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200
2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700


## Derivatives Data

## On-chain Data


#### Coinmetrics

In [ ]:
client = CoinMetricsClient()

# Display metrics available
list_asset_metrics = client.catalog_asset_metrics_v2(assets='btc')
btc_list = list_asset_metrics.to_dataframe()
btc_list

,asset,metric,frequency,min_time,max_time,min_height,max_height,min_hash,max_hash,community
0,btc,AdrActCnt,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
1,btc,AdrBalCnt,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
2,btc,AssetCompletionTime,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
3,btc,AssetEODCompletionTime,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
4,btc,BlkCnt,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
5,btc,CapMVRVCur,1d,2010-07-18 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
6,btc,CapMrktCurUSD,1d,2010-07-18 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
7,btc,CapMrktEstUSD,1d,2019-06-22 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
8,btc,FeeTotNtv,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True
9,btc,FlowInExNtv,1d,2009-01-03 00:00:00+00:00,2026-01-20 00:00:00+00:00,None,None,None,None,True


In [8]:
# Filter for metrics with frequency == '1d'
daily_metrics = btc_list[btc_list['frequency'] == '1d']
daily_metrics_names = daily_metrics['metric'].tolist()

# Print the list
print(f"Total daily metrics: {len(daily_metrics_names)}")
print(daily_metrics_names)

Total daily metrics: 31
['AdrActCnt', 'AdrBalCnt', 'AssetCompletionTime', 'AssetEODCompletionTime', 'BlkCnt', 'CapMVRVCur', 'CapMrktCurUSD', 'CapMrktEstUSD', 'FeeTotNtv', 'FlowInExNtv', 'FlowInExUSD', 'FlowOutExNtv', 'FlowOutExUSD', 'HashRate', 'IssTotNtv', 'IssTotUSD', 'PriceBTC', 'PriceUSD', 'ROI1yr', 'ROI30d', 'ReferenceRate', 'ReferenceRateETH', 'ReferenceRateEUR', 'ReferenceRateUSD', 'SplyCur', 'SplyExNtv', 'SplyExUSD', 'SplyExpFut10yr', 'TxCnt', 'TxTfrCnt', 'volume_reported_spot_usd_1d']


In [16]:
btc_metrics = client.get_asset_metrics(assets = 'btc', metrics = daily_metrics_names,
                                   start_time = '2014-09-17', end_time = '2026-01-17', frequency = '1d')

# Convert to DataFrame directly
btc_coinmetrics = btc_metrics.to_dataframe()

# Convert timestamp to datetime and set 'Date' as index
btc_coinmetrics['time'] = pd.to_datetime(btc_coinmetrics['time'])
btc_coinmetrics.set_index('time', inplace=True)
btc_coinmetrics.index = btc_coinmetrics.index.strftime('%Y-%m-%d')
btc_coinmetrics.index.name = 'Date'
btc_coinmetrics

,asset,AdrActCnt,AdrBalCnt,AssetCompletionTime,AssetEODCompletionTime,BlkCnt,CapMVRVCur,CapMrktCurUSD,CapMrktEstUSD,FeeTotNtv,...,SplyExNtv,SplyExNtv-status,SplyExNtv-status-time,SplyExUSD,SplyExUSD-status,SplyExUSD-status-time,SplyExpFut10yr,TxCnt,TxTfrCnt,volume_reported_spot_usd_1d
Date,,,,,,,,,,,,,,,,,,,,,
2014-09-17,btc,191063,3404363,1614336399,1614336399,186,1.285606,6060691069.13061,<NA>,12.040404,...,417305.412444,reviewed,2020-02-28T09:17:32.260416000Z,190433106.549981,reviewed,2020-02-28T09:17:32.260416000Z,19708906.25,78526,193554,11961580.812249
2014-09-18,btc,190890,3412475,1614336400,1614336400,179,1.205033,5671343316.691176,<NA>,12.215005,...,425198.293926,reviewed,2020-02-28T09:17:32.260416000Z,181508693.188384,reviewed,2020-02-28T09:17:32.260416000Z,19709465.625,76357,192000,29729670.258177
2014-09-19,btc,172255,3418017,1614336400,1614336400,171,1.115123,5234786700.077503,<NA>,10.819493,...,427132.470154,reviewed,2020-02-28T09:17:32.260416000Z,168244870.516347,reviewed,2020-02-28T09:17:32.260416000Z,19710000.0,70093,170839,28559984.972858
2014-09-20,btc,169941,3420176,1614336401,1614336401,176,1.166208,5464482172.793501,<NA>,10.303744,...,428736.201266,reviewed,2020-02-28T09:17:32.260416000Z,176228303.867605,reviewed,2020-02-28T09:17:32.260416000Z,19710550.0,64168,169223,23384116.542328
2014-09-21,btc,200712,3435000,1614336402,1614336402,164,1.136713,5324715500.714623,<NA>,9.086879,...,427755.457953,reviewed,2020-02-28T09:17:32.260416000Z,171275223.389278,reviewed,2020-02-28T09:17:32.260416000Z,19711062.5,57668,238140,15423671.767001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-13,btc,701272,55891967,1768359739,1768359739,135,1.695646,1903740522551.245117,1903718027531.899902,3.019455,...,2474528.818686,flash,2026-01-14T01:54:11.576353000Z,235833727579.620148,flash,2026-01-14T02:23:58.974679000Z,20826394.53125,340763,731900,18052771185.7714
2026-01-14,btc,707589,55884596,1768443975,1768443975,124,1.725258,1938525133766.582764,1938505576191.794434,3.137914,...,2457081.035855,flash,2026-01-15T01:38:44.411306000Z,238444954170.398285,flash,2026-01-15T02:09:50.267798000Z,20826491.40625,326077,681472,19573284071.115601
2026-01-15,btc,717904,55871260,1768531920,1768531920,139,1.698251,1908645921457.237305,1908622521449.100098,3.186324,...,2454717.517114,flash,2026-01-16T01:56:47.857153000Z,234538782907.915253,flash,2026-01-16T02:28:57.257562000Z,20826600.0,375169,792811,15513091732.360901


#### Blockchain.com

In [28]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# Métricas disponíveis na Blockchain.com (gratuitas)
metrics = {
    'n-transactions': 'transactions',
    'n-unique-addresses': 'unique_addresses',
    'hash-rate': 'hash_rate',
    'difficulty': 'difficulty', 
    'miners-revenue': 'miners_revenue',
    'transaction-fees': 'transaction_fees',
    'market-price': 'market_price',
    'total-bitcoins': 'total_bitcoins',
    'mempool-size': 'mempool_size'
}

base_url = "https://api.blockchain.info/charts/"

# Calcular timespan em dias
start = datetime(2011, 1, 1)
end = datetime(2025, 11, 30)
timespan = (end - start).days

print("A obter dados da Blockchain.com...")
all_data = {}

for metric_key, metric_name in metrics.items():
    try:
        url = f"{base_url}{metric_key}"
        params = {
            'timespan': f'{timespan}days',
            'format': 'json',
            'sampled': 'false'
        }

        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            values = data.get('values', [])

            # Converter timestamps para datas
            df_temp = pd.DataFrame(values)
            df_temp['date'] = pd.to_datetime(df_temp['x'], unit='s')
            df_temp = df_temp.rename(columns={'y': metric_name})
            df_temp = df_temp[['date', metric_name]]

            all_data[metric_name] = df_temp
            print(f"✓ {metric_name}: {len(df_temp)} pontos")
        else:
            print(f"✗ {metric_name}: erro {response.status_code}")

        time.sleep(0.5)  # Pausa para não sobrecarregar API

    except Exception as e:
        print(f"✗ {metric_name}: {e}")

# Combinar todos os dataframes
if all_data:
    df_final = None
    for metric_name, df_temp in all_data.items():
        if df_final is None:
            df_final = df_temp
        else:
            df_final = pd.merge(df_final, df_temp, on='date', how='outer')

    df_final = df_final.sort_values('date')

    # Apenas exibir informações, sem salvar CSV
    print(f"\n✓ Dados processados com sucesso!")
    print(f"Período: {df_final['date'].min()} até {df_final['date'].max()}")
    print(f"Dimensões: {len(df_final)} linhas × {len(df_final.columns)} colunas")
    print(f"Colunas disponíveis: {list(df_final.columns)}")
    
    # Mostrar estatísticas básicas
    print(f"\nEstatísticas básicas:")
    print(df_final.describe())
    
    # Mostrar primeiras e últimas linhas
    print(f"\nPrimeiras 5 linhas:")
    print(df_final.head())
    print(f"\nÚltimas 5 linhas:")
    print(df_final.tail())
    
    # Agora df_final está disponível para uso no seu projeto
    # Pode usá-lo diretamente para análise ou modelação
else:
    print("Nenhum dado foi obtido.")
    df_final = pd.DataFrame()  # Dataframe vazio para evitar erros

# O dataframe 'df_final' agora está disponível para uso no seu projeto
# Pode continuar a trabalhar com ele, por exemplo:
# - Criar features adicionais
# - Combinar com dados de outras fontes
# - Treinar modelos de machine learning

A obter dados da Blockchain.com...
✓ transactions: 5444 pontos
✓ unique_addresses: 5432 pontos
✓ hash_rate: 5444 pontos
✓ difficulty: 5444 pontos
✓ miners_revenue: 5447 pontos
✓ transaction_fees: 5444 pontos
✓ market_price: 5448 pontos
✓ total_bitcoins: 801526 pontos
✓ mempool_size: 334278 pontos

✓ Dados processados com sucesso!
Período: 2011-01-28 00:00:00 até 2025-12-27 22:00:00
Dimensões: 1137138 linhas × 10 colunas
Colunas disponíveis: ['date', 'transactions', 'unique_addresses', 'hash_rate', 'difficulty', 'miners_revenue', 'transaction_fees', 'market_price', 'total_bitcoins', 'mempool_size']

Estatísticas básicas:
                                date   transactions  unique_addresses  \
count                        1137138    5444.000000      5.432000e+03   
mean   2019-03-20 14:34:46.208970240  236560.735489      4.160045e+05   
min              2011-01-28 00:00:00     789.000000      9.890000e+02   
25%       2016-03-11 05:26:09.500000   79022.500000      1.730842e+05   
50%    

#### CryptoDataPy

In [50]:
!pip install cryptodatapy --quiet

# Data Preproc

| Column Name | What It Is | Why It's Useful for Your Project |
| :--- | :--- | :--- |
| **`Close`** | The final trading price of BTC for that day. | This is your primary price reference. All returns are calculated from it. |
| **`Volume`** | The total amount of BTC traded during the day. | High volume often confirms the strength of a price move and is correlated with volatility. |
| **`return_1d`** | The percentage change from yesterday's close to today's close: `(Close_t / Close_{t-1}) - 1`. | Easy to interpret (e.g., 0.02 means a 2% gain). It's the basis for calculating **realized volatility** (e.g., the standard deviation of these returns over the last 30 days). |
| **`log_return_1d`** | The logarithmic return: `log(Close_t / Close_{t-1})`. | **This is the standard in quantitative finance.** Log returns are time-additive, which is a very useful mathematical property. They also tend to be more normally distributed than simple returns, which is an assumption in many models. |
| **`intraday_range`** | The absolute difference between the day's high and low prices: `High - Low`. | A direct, raw measure of how much the price fluctuated *within* the day. A large range signifies high intraday volatility and uncertainty. |
| **`norm_intraday_range`** | The intraday range as a percentage of the day's opening price: `(High - Low) / Open`. | This is a **superior feature** to the raw range. By normalizing, you can compare the intraday volatility across different price levels. A $500 range meant something different when BTC was at $20,000 versus when it was at $60,000. This feature adjusts for that. |